# 08 · Toy fields and their statistics

Toy boxes whose statistics are known, built bottom-up: the toy model, its periodic realisations, and their $P$ and
$\xi$ against the exact expectation on the grid (Part A). The statistics measured on skewers through the same boxes
(the flux, its overflux and the density along the skewers) are in notebook 10, `10_skewer_statistics`; the same field
statistics of the cells of a simulation, TNG50-4, in notebook 09, `09_field_statistics_tng`.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
# float32, the package's precision; skewerjax.config.set_precision("float64") for float64 (docs/precision.md)
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)
from skewerjax.core import LineBundle

In [ ]:
plot_dag(('samplers', 'fields', 'field_stats',)); plt.show()

## Concepts

**3D two-point statistics on a grid**. A field $\delta(\mathbf x)$ on
$N_x \times N_y \times N_z$ points with spacings $\Delta_i$ (the units may differ per axis). Then, we have that: 
$\quad L_i = N_i\Delta_i,\quad V = \prod_i L_i,\quad \Delta V = \prod_i \Delta_i,\quad N = \prod_i N_i$.

The DFT, returns the power computed on the modes $k_i = 2\pi m_i/L_i$ with $m_i \in (-N_i/2,\, N_i/2]$
$\longrightarrow \delta_{\mathbf k} = \Delta V\sum_{\mathbf x}\delta(\mathbf x)\,e^{-i\mathbf k\cdot\mathbf x}$.

The estimators, both without the $\mathbf k = 0$ mode are:
$
\quad
\hat P(\mathbf k) = \lvert\delta_{\mathbf k}\rvert^2 / V,\quad \mathrm{and} \quad
\hat\xi(\mathbf r) = \frac{1}{V}\sum_{\mathbf k \neq 0}\hat P(\mathbf k)\,e^{i\mathbf k\cdot\mathbf r}
= \frac{1}{N}\sum_{\mathbf x} d(\mathbf x)\,d(\mathbf x + \mathbf r),\quad \mathrm{with} \quad d = \delta - \frac{1}{N}\sum_{\mathbf x}\delta(\mathbf x),
$

with $\mathbf r$ every minimum-image separation of the periodic grid ($\lvert r_i\rvert \le L_i/2$).

The binning that we are going to employ satisfies the following properties: 
$
\quad
k_\parallel = \lvert k_y\rvert,\qquad k_\perp = \sqrt{k_x^2 + k_z^2},\qquad
\bar P\big(k_\perp^{(a)}, k_\parallel\big) = \frac{1}{N_a}\sum_{\mathbf k \in a}\hat P(\mathbf k),
$
- $k_\parallel$ keeps the native modes $2\pi m/L_y$ (one row each, no binning);
- $k_\perp$ is averaged within given edges, as above, over the $N_a$ modes of bin $a$;
- $\lvert\mathbf k\rvert$ bins, and wedges of $\mu = k_\parallel/\lvert\mathbf k\rvert$ in $\lvert\mathbf k\rvert$ bins,
  in the same way;
- identically in real space, with $r_\parallel = \lvert r_y\rvert$ (native separations $j\Delta_y$),
  $r_\perp = \sqrt{r_x^2 + r_z^2}$, $\lvert\mathbf r\rvert$ and $\mu = r_\parallel/\lvert\mathbf r\rvert$.

**From the cells to the grid: one path for every source.** The fields enter as gas cells (`core.Cells`, here the
cells of the toy grid, `grf_cells`; in notebook 09 the Voronoi cells of TNG50-4): every cell deposits its number of
particles $n_X V$ at its generating point on an $N_{\rm mesh}^3$ mesh with the B-spline scheme `MAS` (PCS) and an
interlaced second mesh shifted by $H/2$; the mesh divided by its mean is $1 + \delta$, and its transform is divided by
the window $W(\mathbf k) = \prod_i[\sin(k_iH/2)/(k_iH/2)]^4$ (Sefusatti et al. 2016). The toy's generating points are
the nodes of the mesh, so there are no alias images, but the scheme's discrete kernel differs from $W$ near
$k_{\max} = \pi/H$: a bias of the measured $P$ of $3\times 10^{-4}$ at $k_{\max}/2$ and $2\times 10^{-2}$ at
$k_{\max}$ (A3 prints it).

**Errors:** the error of the mean over the $n_{\rm real}$ realisations, $\sigma_a/\sqrt{n_{\rm real}}$ per bin.

## The toy model: what `ToyBoxes` generates

**Grid and realisations.** $n_{\rm real}$ periodic cubes of side $L$ and volume $V = L^3$ on $N^3$ cells of side
$dx = L/N$; the modes are $\mathbf k = (2\pi/L)\,\mathbf n$ with integers $-N/2 < n_i \le N/2$. Realisation $i$ is drawn
from the seed `base_seed` $+\,i$ and regenerated whenever it is needed: white noise $w$ on the
grid, $\delta_{\mathbf k} = dV\,\mathrm{FFT}[w]_{\mathbf k}\sqrt{P(k)/dV}$, so that
$\langle\lvert\delta_{\mathbf k}\rvert^2\rangle = V P(k)$, with $\delta_{\mathbf k = 0} = 0$.

**Correlation function and power spectrum** (isotropic: no Kaiser or Fingers-of-God term). The simplest model
with a BAO feature: a power law plus a Gaussian bump at $r_{\rm BAO}$,

$$
\xi(r) = \underbrace{\frac{A_s\,k_0^{1 - n_s}}{2\pi^2}\,
\frac{\Gamma(n_s + 1)\,\sin\big(\pi(n_s + 1)/2\big)}{r^{\,n_s + 2}}}_{\xi_{\rm pl}(r) \equiv \text{power law}}
+ \underbrace{A_{\rm BAO}\big[g(r - r_{\rm BAO}) + g(r + r_{\rm BAO})\big]}_{\text{bump}},\qquad
g(u) = e^{-u^2/2\sigma_{\rm BAO}^2},\qquad A_{\rm BAO} = R_{\rm BAO}A_s,
$$

$$
P(k) = \underbrace{A_s\left(\frac{k}{k_0}\right)^{n_s - 1}}_{\text{power law}}
+ \underbrace{A_{\rm BAO}\,W(k)}_{\text{bump}},\qquad
W^{\rm 3D}(k) = 4\pi\sqrt{2\pi}\,\sigma_{\rm BAO}\,e^{-k^2\sigma_{\rm BAO}^2/2}
\left[\sigma_{\rm BAO}^2\cos(kr_{\rm BAO}) + \frac{r_{\rm BAO}}{k}\sin(kr_{\rm BAO})\right],
$$

$W$ being the exact Fourier transform of the bump (the mirror term $g(r + r_{\rm BAO})$ makes it even).

Note that, regarding the power law term, $\int_0^\infty x^{\mu - 1}\sin x\,dx = \Gamma(\mu)\sin(\pi\mu/2)$ with $\mu = n_s + 1$: The integral
converges for $-1 < \mu < 1$, so **the closed form only holds for $-2 < n_s < 0$**.
Outside that range the continuous $\xi$ of the power law does not exist: the
integral diverges at $k \to 0$ for $n_s \le -2$ and at $k \to \infty$ for $n_s \ge 0$.
In that case, thecode (`PowerLawGRF.correlation`) returns the numerical Hankel transform of $P$ over the $k$ range of the grid instead.

**Validity.** A valid correlation function needs $P(k) \ge 0$ at every mode (Bochner's theorem; equivalently
$\lvert\xi(r)\rvert \le \xi(0)$). $W$ changes sign, so the power law must exceed the troughs of $A_{\rm BAO}W$:
$
\quad
R_{\rm BAO} \le R_{\max} = \min_{\mathbf k \in \text{grid},\ W(k) < 0}\frac{(k/k_0)^{n_s - 1}}{\lvert W(k)\rvert},
$

**The amplitude.** $A_s$ is set by $\sigma_\delta$, the standard deviation of $\delta$ over the grid. At fixed $R_{\rm BAO}$ both terms are proportional to $A_s$, $P(k) = A_s\,p(k)$ with the shape
$p(k) = (k/k_0)^{n_s - 1} + R_{\rm BAO}W(k)$, and the variance of the grid is the sum of $P$ over its modes:

$$
\sigma_\delta^2 = \frac{1}{V}\sum_{\mathbf k \neq 0} P(k) = A_s\,S,\qquad
S = \frac{1}{L^3}\sum_{\substack{\mathbf n \neq 0\\ -N/2 < n_i \le N/2}} p\!\left(\frac{2\pi\lvert\mathbf n\rvert}{L}\right),
\qquad A_s = \frac{\sigma_\delta^2}{S}.
$$

Inputs:

| Symbol | `ToyBoxes` argument | Meaning | Enters |
|---|---|---|---|
| $\sigma_\delta$ | `sigma_delta` | standard deviation of $\delta$ on the grid | $A_s = \sigma_\delta^2/S$ [$\mathrm{cMpc}^3$] |
| $k_0$ | `k_0` | pivot wavenumber [cMpc$^{-1}$] | the shape $p$ |
| $n_s$ | `n_s` | spectral index: $P \propto k^{n_s - 1}$ | the shape $p$ |
| $R_{\rm BAO}$ | `bao_ratio` | $A_{\rm BAO}/A_s$, the height of the bump in $\xi$ per unit $A_s$, at most $R_{\max}$ [cMpc$^{-3}$] | the shape $p$ |
| $\sigma_{\rm BAO}$ | `sigma_bao` | width of the bump [cMpc] | the shape $p$ |
| $r_{\rm BAO}$ | `r_bao` | position of the bump [cMpc] | the shape $p$ |
| $L$ | `box_cMpc` | box side [cMpc] | the modes: $S$ (so $A_s$) and $\xi_{\rm grid}$ |
| $N$ | `n_cells` | cells per side | the modes: $S$ (so $A_s$) and $\xi_{\rm grid}$ |

**Gas.** H I and other species are one gas, cell by cell: $n_{\rm HI} = \bar n_{\rm HI}(1 + \delta)$,
$n_X = r_X\,n_{\rm HI}$ (Si III and Si II), $T_{\rm gas} = T_0(1 + \delta)^{\gamma - 1}$,
and peculiar velocities $\mathbf v = 0$ or linear theory, $\mathbf v_{\mathbf k} = i\,aHf\,
(\mathbf k/k^2)\,\delta_{\mathbf k}$. The remaining `ToyBoxes` inputs set this gas:
`n_bar_HI` ($\bar n_{\rm HI}$, which sets $\langle T_{{\rm Ly}\alpha}\rangle$), `metal_to_hi` ($r_X$ per metal species), `T0`,
`gamma_minus_1`, `velocity` (`"zero"`, the default, or `"linear"`), `f_growth` ($f$, for `"linear"` only), and `z`,
`Omega_m` ($a$ and $H$ of a flat cosmology: the Hubble flow, the velocity axis of the skewers); `n_real` and `base_seed`
choose the realisations.

**Positivity, $1 + \delta > 0$.** A Gaussian $\delta$ is unbounded, but a density $n = \bar n(1 + \delta)$ must be
positive. The function `ToyBoxes.fields` raises an error if any cell
has $\delta \le -1$. One cell falls below $-1$ with probability $\Phi(-1/\sigma_\delta)$ ($\Phi$ the standard normal
distribution function), and a box has $N^3$ cells, so about $N^3\,\Phi(-1/\sigma_\delta)$ of them do: a
finer grid needs a smaller $\sigma_\delta$.

In [ ]:
import time
from fractions import Fraction
from skewerjax.core import phys
from skewerjax.simulations import PowerLawGRF, ToyBoxes
from skewerjax.summary_stats import bao_k_edges, bao_maxima, box_grid, field_statistics, xi_chi2, xi_corner, xi_subset_offsets

In [ ]:
REALISATIONS = None  # None: all of them; e.g. [0, 1] for a quick check
L_BOX = 320
METAL_TO_HI = {"SiIII": 1 / 10, "SiII": 1 / 20}   # n_X = r_X n_HI: Si III; Si II an illustrative toy choice, half of Si III
boxes = ToyBoxes(box_cMpc=L_BOX, n_s = -0.4, n_cells=320, n_real=12, base_seed=137, bao_ratio=0.0002, sigma_bao=1.0, r_bao=147, metal_to_hi=METAL_TO_HI, velocity="linear", z = 2.2)
N_MESH = boxes.n_cells   # the mesh of the field statistics: the toy grid's own nodes
MAS = "pcs"              # mass assignment of the cells on the mesh (PCS: 4 nodes per axis)
INTERLACE = True         # a second mesh shifted by H / 2 (the odd alias images cancel)


In [ ]:
# The values of the configuration (the text refers to them by their symbols)
from scipy.stats import norm
g = boxes.grf()
idx = np.arange(boxes.n_real) if REALISATIONS is None else np.asarray(REALISATIONS)
var_bao = 1.0 - float(PowerLawGRF(g.A_s, g.k_0, g.n_s, L=g.L, N=g.shape).variance() / g.variance())
S_shape = boxes.sigma_delta**2 / boxes.A_s                                       # the mode sum of the shape p(k), A_s = 1
print(f"dx = {boxes.dx:.4g} cMpc ({boxes.dx * boxes.aH:.1f} km/s); R_BAO = {boxes.bao_ratio:g} <= R_max = {boxes.bao_ratio_max:.4g} cMpc^-3 (P >= 0 at every mode of the grid): A_BAO = {g.A_bao:.4g}, A_s = {g.A_s:.4g} cMpc^3")

## Part A — Toy fields and their statistics

### A1 · The fields and their bins

**Where the statistics are computed.** The $P$ and $\xi$ of every realisation are computed in the `field_statistics(...)`
cell below: the cells of each realisation are generated, deposited on the mesh (`N_MESH`, `MAS`, `INTERLACE`),
measured and discarded in one pass.

The grid fixes the ranges where a bin holds every direction:

- $\lvert\mathbf k\rvert$ from $k_{\min} = 2\pi/L$ to $k_{\max} = \pi/dx$. Inside the sphere of radius $\pi/dx$ a shell of $\lvert\mathbf k\rvert$ has modes in every direction; beyond it only the corners of the cube of modes remain.
- $\lvert\mathbf r\rvert$ from $r_{\min} = dx$ to $r_{\max} = L/2$. Beyond half the box only the separations towards the corners of the cube remain (minimum image).

**The bins** are explicit arrays, defined in the next cell and passed to `field_statistics`:

- `K_EDGES`, the bins of $P(\lvert\mathbf k\rvert)$. One bin per mode up to $k_1$; then bins of
  `dk_factor` times the fundamental mode $k_f = 2\pi/L$, up to $k_2 \simeq 3/\sigma_{\rm BAO}$; then logarithmic bins from $k_2$ up to $k_{\max}$.
  `K_EDGES` are also the $k_\perp$ columns of the $(k_\perp, k_\parallel)$ map.
- `R_EDGES`, the bins of $\xi(\lvert\mathbf r\rvert)$: width $dx$, from 0 to $L/2$. The first bin holds $r = 0$ only, the variance, which the figures leave out. `R_EDGES`
  are also the $\lvert\mathbf r\rvert$ bins of the wedges of $\mu$ and the $r_\perp$ columns of the
  $(r_\perp, r_\parallel)$ map.

In [ ]:
L, dx = boxes.box_cMpc, boxes.dx
K_EDGES, K_INFO = bao_k_edges(2 * np.pi / L, np.pi / dx, 3.0 / boxes.sigma_bao, k_1=None, dk_factor=0.4, n_log=12, info=True)  # |k|: per mode to k_1, dk_factor k_f to k_2 = 3 / sigma_BAO, log to pi / dx
R_EDGES = np.arange(0, L / 2 + dx, dx)                                                                                         # |r| (and the mu wedges, the r_perp columns): width dx to L / 2

In [ ]:
GRID_A = box_grid(N_MESH, L, k_edges=K_EDGES, r_edges=R_EDGES)                                # the mesh and its bins (map rows: the mesh's own modes)
res = field_statistics((boxes.cells(int(i)) for i in idx), GRID_A, species="HI",               # the cells of every realisation through the mesh (A3),
                       mas=MAS, interlace=INTERLACE, power=g.power,                            # computed now, against the expectation with and
                       power_nw=lambda k: g.power(k, bao=False))                               # without the BAO bump
m1 = res["min_one_plus_delta"]
print(f"{res['n_real']} realisations: min(1 + delta) = {m1.min():.3f} (realisation {idx[m1.argmin()]}); 1 + delta > 0 in {np.sum(m1 > 0)} of {m1.size}")
full, ax_A = res["limits"], res["axes"]
print(f"|k| from k_min = 2 pi / L = {full['k_min']:.4f} to k_max = pi / dx = {full['k_max']:.3f} cMpc^-1")
print(f"|r| from r_min = dx = {full['r_min']:.3f} to r_max = L / 2 = {full['r_max']:.1f} cMpc")
cap = " (capped at k_max)" if K_INFO["k_2"] < K_INFO["k_w"] else ""
print(f"|k| bins: {K_INFO['n_mode']} of one mode up to k_1 = {K_INFO['k_1']:.4f}, {K_INFO['n_step']} of dk = dk_factor k_f = "
      f"{K_INFO['dk']:.4f} up to k_2 = {K_INFO['k_2']:.3f}{cap} (factor / sigma_BAO = {K_INFO['k_w']:.3f}), "
      f"{K_INFO['n_log']} logarithmic from k_2 up to {ax_A['k_edges'][-1]:.3f} cMpc^-1")
print(f"|r| bins: {ax_A['r_edges'].size - 1} of width {np.diff(ax_A['r_edges']).max():.3f} cMpc from {ax_A['r_edges'][0]:g} to {ax_A['r_edges'][-1]:.1f} cMpc; every bin inside the full coverage")

### A2 · The fields of one realisation

Realisation 0 in the plane of the first cell layer along the third axis (the cells with index $i_z = 0$), which holds
the first row of skewers: $x$ across and $y$, the line of sight of those skewers, upwards. The columns
(H I, Si III, Si II) differ only by the factors $r_X$ of the densities; the temperature (not drawn) is a function of the density of the top row.
The bottom row shows the velocity along the skewers, $v_y$ (colour), and $(v_x, v_y)$ as arrows every `ARROW_EVERY`
cells; with zero velocities (`velocity="zero"`) it is empty.

In [ ]:
from skewerjax.utils.plotting import plot_toy_fields
ARROW_EVERY = max(1, boxes.n_cells // 32)                                    # about 32 arrows across the slice
fields0 = boxes.fields(0)
fig, axs, info = plot_toy_fields(fields0, L_BOX, z_index=0, arrow_every=ARROW_EVERY)
add_panel_letters(axs, labels="grid")
plt.show()

### A3 · The statistics of the realisations

Each realisation is measured on the bins of A1 (`K_EDGES`, `R_EDGES`).

**The expectation (black line).**

1. The generator draws every mode $\mathbf k$ of the box with $\langle\hat P(\mathbf k)\rangle = P(k)$, and the
   correlation at every separation $\mathbf r$ of the grid is, on average, $\xi_{\rm grid}(\mathbf r)$.
2. A bin $a$ of $\lvert\mathbf k\rvert$ holds the $N_a$ modes of the box that fall in it, so the expectation of the bin
   average is the input averaged over exactly those modes,
   $\bar P_a = N_a^{-1}\sum_{\mathbf k \in a} P(\lvert\mathbf k\rvert)$.
3. Likewise, a bin $a$ of $\lvert\mathbf r\rvert$ holds the $N_a$ separations of the grid that fall in it, and
   $\bar\xi_a = N_a^{-1}\sum_{\mathbf r \in a}\xi_{\rm grid}(\mathbf r)$, with
   $\xi_{\rm grid}(\mathbf r) = V^{-1}\sum_{\mathbf k \neq 0} P(k)\cos(\mathbf k\cdot\mathbf r)$.

**Continuous $P(k)$ and $\xi(r)$ (purple line).** They are the infinite-volume curves of the same
spectrum, over the full-coverage range, and a finite periodic grid does not measure them:

- **The integral constraint: no $k = 0$ mode.** $\xi_{\rm grid}$ integrates to zero over the box (Peebles 1980; Landy &
  Szalay 1993; Bagla & Prasad 2006, arXiv:astro-ph/0601320; de Mattia & Ruhlmann-Kleider 2019, arXiv:1904.08851).
  Leaving out the cube of modes (the kernel $w$ of the maps below), Poisson summation gives

  $$
  \xi_{\rm grid}(\mathbf r) = \big[\xi(r) - \bar\xi_V\big]
  + \sum_{\mathbf n \neq 0}\big[\xi(\lvert\mathbf r + L\mathbf n\rvert) - \langle\xi(\lvert\,\cdot + L\mathbf n\rvert)\rangle_V\big],
  \qquad \bar\xi_V = \frac{1}{V}\int_{[-L/2,\,L/2]^3}\xi(\lvert\mathbf x\rvert)\,d^3x,
  $$

  with $\langle\cdot\rangle_V$ the mean over the box. It is finite even though $P(0)/V$ and the image sum diverge separately
  for a red spectrum: each bracket subtracts the mean of its own image. `integral_constraint(xi, L)` gives $\bar\xi_V$, a 1D
  integral with the exact area of the sphere inside the cube, and $\xi - \bar\xi_V$ is the dashed purple curve of (b).
  What remains between it and the expectation is the image fluctuation near $L/2$ (the sum over $\mathbf n \neq 0$), the
  cube kernel at a few $dx$, and the binning (printed below the figure).
- **Periodic images.** $\xi_{\rm grid}(\mathbf r)$ adds the correlation of the images $\mathbf r + L\mathbf n$.
- **Averaging within bins.** $P$ and $\xi$ vary inside a bin, and the few modes of a low-$k$
  bin sample it unevenly.

The binned expectation includes all of this, so it is exact bin by bin; the continuous curves are for reference only.

**The error bars** are the error of the mean over the $n_{\rm real}$ realisations, bin by bin:
$\sigma_a/\sqrt{n_{\rm real}}$, with $\sigma_a$ the standard deviation (with $n_{\rm real} - 1$) of the $n_{\rm real}$
values of bin $a$. The residual panels divide by it.

In (b) the points can sit on one side of the expectation over a whole range of $r$, by more than their
error bars. This is one ordinary fluctuation, not a bias:

- Every bin of $\xi$ is a sum over the same modes,
  $\hat\xi(\mathbf r) = V^{-1}\sum_{\mathbf k}\hat P(\mathbf k)\cos(\mathbf k\cdot\mathbf r)$, and with a red spectrum
  ($P(k)$ decreasing with $k$, here $n_s - 1 < 0$, so most of the power is on large scales) the few largest-scale modes
  carry most of the signal. When their random amplitudes come out a high (or low),
  every bin moves up (or down) together. The error bars show the scatter of each bin alone, so they cannot show this:
  several points off by about $1\sigma$ in the same direction are one fluctuation, not several.
- The exact covariance $C$ of the bins is dominated by its first eigenvector. The $\chi^2$ that accounts for it,
  $\chi^2 = \mathbf d^{\rm T}(C/n_{\rm real})^{-1}\mathbf d$ with $\mathbf d = \bar\xi - \xi_{\rm grid}$, is close to
  the number of bins, over all the bins and over the window of the BAO peak, `XI_WINDOW`.
- **More realisations shrink it.** For disjoint subsets of realisations, the offset of $\xi$ in `XI_WINDOW` changes sign from subset to subset and stays within its standard
  deviation, which falls as $1/\sqrt n$.
- **$P$ shows no such effect**, because its bins hold different modes, which are independent.

The solid purple continuous $\xi(r)$ lies above both by the integral constraint $\bar\xi_V$; the dashed $\xi - \bar\xi_V$
follows the expectation up to what remains above. The purple $P(k)$ runs through the binned expectation, except in the first
bins, which hold a few modes each. The dotted grey lines of (a) and its residuals are the maxima of the model's bump term $A_{\rm BAO}W(k)$
(`bao_maxima`), and those of (b) and its residuals $r_{\rm BAO}$.

In [ ]:
from skewerjax.utils.plotting import plot_isotropic_recovery, plot_kpar_kperp_maps, plot_rpar_rperp_maps
K_MARKS = {"$k_1$": K_INFO["k_1"], "$k_2$": K_INFO["k_2"]}  # where the bins of K_EDGES change (A1)
BAO_K = bao_maxima(np.pi / dx, window=g.bao_window, r_bao=boxes.r_bao)  # the positive maxima of the bump term A_BAO W(k) up to pi / dx [cMpc^-1]
fig, axs, rec = plot_isotropic_recovery(
      res, quantity="xi", r_bao=boxes.r_bao, grf=g, k_edges=K_EDGES, r_edges=R_EDGES,
      k_marks=K_MARKS, y_scale="symlog", k_scale="linear", bao_k=BAO_K, show_analytic=False
)
add_panel_letters(axs[0])                                              # (a) P, (b) r^2 xi; the residual panels below them
plt.show()
xi_bar = rec["xi_bar_V"]                                               # the integral constraint: the mean of xi over the box
r_iso = res["axes"]["r_iso"]
ok_ic = np.isfinite(res["theory"]["xi_iso"]) & (r_iso > 0)
dev = np.abs(np.asarray(g.correlation(r_iso[ok_ic])) - xi_bar - res["theory"]["xi_iso"][ok_ic])
print(f"integral constraint: xi_bar_V = {xi_bar:.4e}; max |(xi - xi_bar_V) - expectation|: r < L / 4 {dev[r_iso[ok_ic] < L / 4].max():.2e}, r < L / 2 {dev.max():.2e}; "
      f"r >= 6 H and r < L / 4 {dev[(r_iso[ok_ic] >= 6 * dx) & (r_iso[ok_ic] < L / 4)].max():.2e}, r >= 6 H and r < L / 2 {dev[r_iso[ok_ic] >= 6 * dx].max():.2e}")
print(f"BAO maxima in k (positive maxima of W, up to pi / dx): {BAO_K.size}, the first {np.round(BAO_K[:3], 3)} cMpc^-1")
SUBSETS = (4, 8, 16)                                                   # disjoint subsets of realisations (reading below)
XI_WINDOW = (2 * boxes.r_bao / 3, boxes.r_bao + 4 * boxes.sigma_bao)   # the r range of the offset [cMpc]

**How far the mesh can be trusted.** The mesh has spacing $H$ and resolves modes up to $k_{\max} = \pi/H$ along each axis. Depositing the cells with PCS and dividing by its continuous window is exact only for randomly placed points; the toy's generating points sit exactly on the nodes, so the deposit leaves a residual bias, `node_transfer` $-\,1$. It is below $10^{-3}$ in $P$ for $k < k_{\max}/2$ and below $0.3\sigma$ in $\xi$ for $r \ge 6H$: the trusted range. Closer to $k_{\max}$, and above all in the corners of the cube of modes (up to a factor 17 at $\sqrt3\,k_{\max}$), the bias grows, and $\xi$ at $r < 6H$, a sum over every mode, inherits it. The table gives the bias per range of $k/k_{\max}$ next to the measured offset $\bar P/P_{\rm exp} - 1$ and its error; the $\chi^2$ of $\xi$ is given over every bin and over $r \ge 6H$.

In [ ]:
from skewerjax.summary_stats import grid_power, node_transfer
P_modes = grid_power(g.power, GRID_A.shape, GRID_A.L)
expected = GRID_A.theory(node_transfer(N_MESH, L, mas=MAS, interlace=INTERLACE) * P_modes)["P_iso"]  # the exact mean through the mesh
k_max = res["k_nyq"]                                                    # pi / H: the largest mode of the mesh along an axis
x = res["axes"]["k_iso"] / k_max
bias, off = expected / res["theory"]["P_iso"] - 1, res["P_iso"] / res["theory"]["P_iso"] - 1
err = res["P_iso_err"] / res["theory"]["P_iso"]
R_TRUST = res["trusted"]["r_min"]                                       # xi: beyond the reach of the PCS kernel (6 H)
print(f"trusted range of the mesh path: k < k_max / 2 = {k_max / 2:.3f} cMpc^-1 (|bias| < {np.nanmax(np.abs(bias[x < 0.5])):.1e}); "
      f"r >= 6 H = {R_TRUST:.2f} cMpc")
def chi2_trusted(key):                                                  # chi2/dof per bin of xi over r >= R_TRUST (diagonal), per row
    a, r = res["axes"], {"xi_map": res["axes"]["rpar_map"] ** 2 + res["axes"]["rperp_map"] ** 2, "xi_mu": res["axes"]["r_mu"] ** 2}[key]
    pull = (res[key] - res["theory"][key]) / res[key + "_err"]
    ok = np.isfinite(pull) & (r >= R_TRUST**2)
    return np.array([np.mean(p[o] ** 2) for p, o in zip(pull, ok)]) if key == "xi_mu" else float(np.mean(pull[ok] ** 2))
print("  k / k_max     bias (node_transfer)   measured P / P_exp - 1   error")
for lo, hi in ((0, 0.1), (0.1, 0.25), (0.25, 0.5), (0.5, 0.75), (0.75, 1.0)):
    s = (x >= lo) & (x < hi) & np.isfinite(off)
    print(f"  {lo:4.2f}-{hi:4.2f}      {np.mean(bias[s]):+.1e}               {np.mean(off[s]):+.1e}                {np.mean(err[s]):.1e}")

**The $(k_\perp, k_\parallel)$ plane.** The same statistics resolved across ($x$ axis) and
along ($y$ axis) the line of sight, on the grid's own modes and separations.

In [ ]:
figs, axs, chi2_P = plot_kpar_kperp_maps(
    res, kperp_edges=K_EDGES, n_sigma=1, k_marks=K_MARKS, kpar_scale="linear", kperp_scale="linear", kms_per_cMpc=boxes.aH
)  # expected, measured
add_panel_letters(axs)
plt.show()

**The $(r_\perp, r_\parallel)$ plane.** The same statistics resolved across ($x$ axis) and
along ($y$ axis) the line of sight, on the grid's own modes and separations. The white rays are the boundaries of the wedges of $\mu$
of the wedge figure.

**The bins.** One row per separation of the grid along the line of sight, $r_\parallel = j\,dx$ from 0 to $L/2$, each
centred on its separation. Across, every pair of grid points has $r_\perp = \sqrt{\Delta x^2 + \Delta z^2}$ (minimum-image
periodic separations) and is averaged into the column of `R_EDGES` that contains it; the first column, of width $dx$
from 0, holds only $r_\perp = 0$, the pairs on the same column of cells.

**Why the grid's $\xi$ is not isotropic.** The toy field is isotropic, yet its expectation on the grid is not a
function of $\lvert\mathbf r\rvert$ alone.

(i) The grid holds the modes $\mathbf k = 2\pi\mathbf n/L$ of the cube $\mathcal K$, $\lvert k_i\rvert \le \pi/dx$, so
$
\quad \longrightarrow \quad \xi_{\rm grid}(\mathbf r) = \frac{1}{V}\sum_{\mathbf k \in \mathcal K,\ \mathbf k \neq 0} P(k)\,e^{i\mathbf k\cdot\mathbf r}.
$

(ii) Poisson summation, $V^{-1}\sum_{\mathbf k \in 2\pi\mathbb Z^3/L} f(\mathbf k)\,e^{i\mathbf k\cdot\mathbf r} =
\sum_{\mathbf n \in \mathbb Z^3}\tilde f(\mathbf r + L\mathbf n)$ with $\tilde f$ the inverse Fourier transform of $f$,
applied to $f = P\,\mathbb 1_{\mathcal K}$, turns the discrete sum into a sum over periodic images:

$$
\xi_{\rm grid}(\mathbf r) = \sum_{\mathbf n \in \mathbb Z^3}(\xi * w)(\mathbf r + L\mathbf n) - \bar\xi_{\rm box},\qquad
w(\mathbf x) = \prod_i \frac{\sin(\pi x_i/dx)}{\pi x_i},
$$

with $w$ the transform of the cube $\mathcal K$, an anisotropic kernel of width $\sim dx$, and
$\bar\xi_{\rm box} = P(0)/V$, the removed $\mathbf k = 0$ mode. For a red
spectrum each of the two terms diverges on its own; their difference, the sum without $\mathbf k = 0$, is finite.

(iii) **Effect 1, the images.** The nearest image of $\mathbf r = r\hat{\mathbf s}$ lies at
$d_1(\hat{\mathbf s}) = \min_{\mathbf n \neq 0}\lvert r\hat{\mathbf s} + L\mathbf n\rvert$.

- Along an axis, $\hat{\mathbf s} = \hat{\mathbf y}$: $d_1 = L - r$, which at $r = L/2$ is as close as $\mathbf r$ itself.
- Along a body diagonal, $\hat{\mathbf s} = (1, 1, 1)/\sqrt3$, at $r = L/2$: $r\hat{\mathbf s} = (L/2\sqrt3)(1, 1, 1)
  \simeq 0.289L\,(1, 1, 1)$, and the nearest image, $\mathbf n = (-1, 0, 0)$, lies at
  $L\sqrt{(1 - 0.289)^2 + 2\,(0.289)^2} = L\sqrt{0.506 + 0.167} \simeq 0.82L$ (printed below).

So the image term, $\xi(d_1)$, depends on the direction.

(iv) **Effect 2, the cube kernel.** The modes stop at $\pi/dx$ along an axis but at $\sqrt3\,\pi/dx$ along a body
diagonal, so $w$ smooths $\xi$ differently along the axes and along the diagonals, on scales of a few $dx$.

In [ ]:
figs, axs, chi2_xi = plot_rpar_rperp_maps(
      res, rperp_edges=R_EDGES, r_bao=boxes.r_bao, n_sigma=0.2, quantity="r2xi",
      color_scale="symlog", kms_per_cMpc=boxes.aH, levels=[5*10**-5]
      )# expected, measured
add_panel_letters(axs)
plt.show()
print(f"chi2/dof per bin: xi map {chi2_xi:.2f}, r >= 6 H: {chi2_trusted('xi_map'):.2f}; {ax_A['rpar_edges'].size - 1} rows (r_par = j dx, j = 0 .. N / 2), "
      f"{ax_A['rperp_edges'].size - 1} columns of r_perp (R_EDGES)")
# The ring and what lies beneath it, wedge by wedge at r_BAO: the bump = xi - xi_nw (xi_nw: the expectation of the
# power law alone) and the background xi_nw
a = res["axes"]
j = np.nanargmin(np.abs(a["r_mu"] - boxes.r_bao), axis=1)   # empty bins of a narrow wedge are NaN
w = np.arange(len(j))
th, nw = res["theory"]["xi_mu"][w, j], res["theory_nw"]["xi_mu"][w, j]
meas, err = res["xi_mu"][w, j] - nw, res["xi_mu_err"][w, j]
print(f"at r = r_BAO, in units of A_BAO = {g.A_bao:.3g}:")
print("  wedge of mu      bump expected   bump measured     background xi_nw")
for k in w:
    print(f"  {a['mu_edges'][k]:.2f} - {a['mu_edges'][k + 1]:.2f}       {(th[k] - nw[k]) / g.A_bao:6.3f}      "
          f"{meas[k] / g.A_bao:6.2f} +- {err[k] / g.A_bao:4.2f}      {nw[k] / g.A_bao:+6.2f}")
s_diag = np.ones(3) / np.sqrt(3)                                          # a body diagonal (text below)
n_img = np.array(np.meshgrid(*[np.arange(-1, 2)] * 3, indexing="ij")).reshape(3, -1).T
n_img = n_img[np.any(n_img != 0, axis=1)]                                  # the nearest periodic images, n != 0
d_1 = lambda s: np.min(np.linalg.norm(0.5 * s + n_img, axis=1))            # [L], for r = L / 2 along s
print(f"nearest periodic image of r = L / 2: along an axis at {d_1(np.array([0.0, 1.0, 0.0])):.3f} L, along a body diagonal "
      f"at {d_1(s_diag):.3f} L")
k_max = np.pi / boxes.dx
print(f"|P_BAO| at k_max = pi / dx = {k_max:.2f} cMpc^-1, relative to its value at the fundamental mode: "
      f"{abs(float(g.bao_power(k_max) / g.bao_power(2 * np.pi / L_BOX))):.1e}")

**The corner $r_\parallel \simeq L/2$, $r_\perp < 10$ cMpc.** The measured map shows a positive patch there that the
expected one does not have. It is a fluctuation of these 6 realisations, not a bug or a bias of the estimator (next cell, and
the scratch runs of stem 20261104):

- **Its size.** The mean of $r^2(\hat\xi - \xi_{\rm grid})$ over the 27 cells of the corner is $+0.06$ cMpc$^2$, with an exact
  Gaussian error of the mean of 6 of $0.053$: $1.1\sigma$, most of it from one realisation ($+0.23$). The cell at $(0, L/2)$ is $1.6\sigma$ ($0.15$ against $0.093$).
- **It is not the mesh.** The exact FFT of the same fields gives the same patch, within $10^{-3}$, and so does every scheme
  (NGP, CIC, TSC, PCS, interlaced or not, within $0.02$). With 48 realisations the corner is $+0.01 \pm 0.02$, and disjoint
  subsets of 6 change sign ($+0.06, +0.01, -0.04, +0.01, +0.06, -0.12, +0.09, +0.04$).
- **Why it shows there.** The weight $r^2$ is at its largest, and a lag with $2\mathbf r = 0$ modulo $L$ has twice the variance:
  $\mathbf r \equiv -\mathbf r$, $\hat\xi(L\hat{\mathbf y}/2) = V^{-1}\sum_{\mathbf k}\hat P(\mathbf k)(-1)^{m_y}$. So
  $\sigma(r^2\xi)$ of one lag on the line of sight, for the mean of 6, grows from $0.016$ at $L/4$ to $0.093$ at $L/2$.
- **What is systematic is much smaller.** Through the mesh at the nodes, $\hat\xi - \xi_{\rm grid} = V^{-1}\sum_{\mathbf k}(T - 1)\hat P\cos(\mathbf k\cdot\mathbf r)$,
  $T$ = `node_transfer`, alternates as $(-1)^j$ along $r_\perp = 0$ (the kink of $T$ across $k_y = \pi/H$): $4\times 10^{-6}$ beyond
  $j \simeq 50$, $\pm 0.02$ cMpc$^2$ in $r^2\xi$ at $L/2$, a quarter of the error and of alternating sign, unlike the patch. Zeroing the
  planes $k_i = \pm k_{\max}$ is not a remedy: it removes $\sum_{\rm plane}P/V = 9.4\times 10^{-5}$ from every row with sign $(-1)^j$, $\pm 0.5$ cMpc$^2$ at $L/2$.
- **The bookkeeping is right.** $j = N/2$ is one row, with half the lags of the others, in the measurement and the expectation
  alike; its value equals the pair count. At $k_i = k_{\max}$ numpy keeps $k_i = -\pi/H$, where the phase of interlacing is $-i$; for points on
  the nodes the shifted PCS mesh vanishes there, so $T = 1.03$ on the axes.


In [ ]:
C8 = xi_corner(res, grid=GRID_A, power=g.power, r_perp_max=10.0)                # the corner of the map: r_par within 2 dx of L / 2, r_perp < 10 cMpc
th_T = GRID_A.theory(node_transfer(N_MESH, L, mas=MAS, interlace=INTERLACE) * P_modes)["xi_map"]   # the exact mean through the mesh
a_c = res["axes"]
print(f"the corner (r_par = {C8['rows'][0]:.1f} .. {C8['rows'][-1]:.1f}, r_perp < 10 cMpc, {C8['rows'].size * C8['columns'].size} cells): mean of r^2 (xi - xi_grid) = {C8['offset']:+.3f} cMpc^2, "
      f"exact Gaussian error of the mean {C8['sigma']:.3f} ({C8['offset'] / C8['sigma']:.1f} sigma); scatter {C8['error']:.3f}; each realisation {np.round(C8['each'], 2)}")
print(f"disjoint subsets of 3 realisations: {C8['each'][:3].mean():+.3f}, {C8['each'][3:6].mean():+.3f} cMpc^2")
r2_last = a_c["rpar_map"][-1, 0] ** 2
print(f"the column r_perp = 0: (-1)^j part over the last 16 rows {C8['alternating']:+.3f} +- {C8['alternating_err']:.3f} cMpc^2; expected through the mesh at r_par = L / 2: "
      f"r^2 (T - 1) P = {r2_last * (th_T[-1, 0] - res['theory']['xi_map'][-1, 0]):+.3f} cMpc^2")


**Wedges.** The direction of a separation (or of a wavevector) is measured by:
$
\quad
\mu = \frac{r_\parallel}{\lvert\mathbf r\rvert} = \cos\theta,
\quad
$
the cosine of its angle $\theta$ to the line of sight: $\mu = 1$ along the line of sight, $\mu = 0$ across it.

The expectation of a wedge $w$ (an interval of $\mu$ of width $\Delta\mu$), step by step:

1. In the continuum, the correlation depends on the separation and its direction, $\xi(r, \mu)$, and the wedge average
   is $\xi_w(r) = \Delta\mu^{-1}\int_{\mu \in w}\xi(r, \mu)\,d\mu$. The toy field is isotropic, $\xi(r, \mu) = \xi(r)$,
   so every wedge would be the same curve.
2. On the grid, a wedge is the average over the separations of the grid with $\lvert\mathbf r\rvert$ in a bin $a$ of
   `R_EDGES` and $\mu = \lvert r_y\rvert/\lvert\mathbf r\rvert$ in the wedge $w$ (the same `MU_WEDGES` as the rays of the
   maps), and its expectation averages $\xi_{\rm grid}$ over the same separations:
   $\bar\xi_{a w} = N_{aw}^{-1}\sum_{\mathbf r \in a,\,\mu \in w}\xi_{\rm grid}(\mathbf r)$.
3. $\xi_{\rm grid}$ is not isotropic (the explanation above), so the dashed lines differ between wedges: the wedge near
   $\mu = 1$ departs most, near $L/2$.

In [ ]:
from skewerjax.utils.plotting import plot_xi_mu_wedges
fig, axs, chi2_rmu = plot_xi_mu_wedges(res, r_edges=R_EDGES, r_bao=boxes.r_bao, quantity="xi", y_scale="symlog")
plt.show()
print("chi2/dof per wedge (diagonal):", np.round(chi2_rmu, 2), "; r >= 6 H:", np.round(chi2_trusted("xi_mu"), 2))